# Vector Graph Memory - Playground

This notebook demonstrates the Vector Graph Memory system with a job search use case.

## Setup and Imports

In [12]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from gremlin_python.driver import client as gremlin_client
from pydantic_ai.embeddings.openai import OpenAIEmbeddingModel

from src.vgm import MemoryAgent, MemoryConfig, MemoryTriggerConfig, AuditConfig

# Load environment variables
load_dotenv()

print("✓ Imports successful")

✓ Imports successful


## Database Connections

Make sure you've started the databases with `docker compose up -d`

In [13]:
# Qdrant connection
qdrant_host = os.getenv('QDRANT_HOST', 'localhost')
qdrant_port = int(os.getenv('QDRANT_HTTP_PORT', '6333'))

qdrant = QdrantClient(host=qdrant_host, port=qdrant_port)
print(f"✓ Connected to Qdrant at {qdrant_host}:{qdrant_port}")

✓ Connected to Qdrant at localhost:6123


In [14]:
# JanusGraph connection
janusgraph_host = os.getenv('JANUSGRAPH_HOST', 'localhost')
janusgraph_port = int(os.getenv('JANUSGRAPH_PORT', '8182'))

janus = gremlin_client.Client(
    f'ws://{janusgraph_host}:{janusgraph_port}/gremlin',
    'g'
)
print(f"✓ Connected to JanusGraph at {janusgraph_host}:{janusgraph_port}")

✓ Connected to JanusGraph at localhost:8481


## Initialize Memory Agent

Configure the agent with:
- Embedding model for vector search
- LLM model for agent reasoning
- Memory configuration (use case, threshold, project)
- Audit logging (JSONL or MongoDB)

In [15]:
# Configure embedding and LLM models
embedding_model = OpenAIEmbeddingModel('text-embedding-3-small')
llm_model = 'openai:gpt-4o-mini'

# Define system prompt
system_prompt = """
You are a helpful assistant for managing job search activities.
You help track job applications, companies, contacts, and interactions.
"""

# Configure memory behavior
memory_config = MemoryConfig(
    use_case_description="Track job search: companies, job postings, contacts, and interactions",
    memory_threshold_description="Store specific job postings, company information, contact details, and meaningful interactions",
    project_id="job_search_2026",
    similarity_threshold=0.85,
)

# Configure memory triggers (when to check for items to remember)
trigger_config = MemoryTriggerConfig(
    mode="phrase",
    trigger_phrase="save this to memory",
)

# Configure audit logging (using JSONL by default)
audit_config = AuditConfig(
    backend="jsonl",
    log_dir="~/.vgm/logs",
)

# Create the agent
agent = MemoryAgent(
    qdrant_client=qdrant,
    janus_client=janus,
    embedding_model=embedding_model,
    llm_model=llm_model,
    system_prompt=system_prompt,
    memory_config=memory_config,
    trigger_config=trigger_config,
    audit_config=audit_config,
)

print("✓ Memory Agent initialized")
print(f"  - Embedding: text-embedding-3-small")
print(f"  - LLM: {llm_model}")
print(f"  - Project: {memory_config.project_id}")
print(f"  - Trigger: {trigger_config.mode}")
print(f"  - Audit: {audit_config.backend}")

✓ Memory Agent initialized
  - Embedding: text-embedding-3-small
  - LLM: openai:gpt-4o-mini
  - Project: job_search_2026
  - Trigger: phrase
  - Audit: jsonl


## Usage Examples

### Example 1: Basic Conversation

In [16]:
result = agent.run("What tools do you have available for managing my job search?")
print(result.output)

I have the following tools available for managing your job search:

1. **Search Memory**: I can search the memory for semantically similar content, allowing us to track existing applications, companies, contacts, and interactions.

2. **Propose Memory Addition**: I can propose adding new content to memory, such as job postings, company information, or contact details, but it requires your confirmation before anything is stored.

3. **Get Memory Context**: I can explore relationships between stored information via graph traversal, which can help us understand connections between companies, jobs, and contacts.

These tools can help you keep track of your job applications and associated activities effectively. Let me know how you'd like to use them!


### Example 2: Adding Information to Memory

The agent will propose adding items to memory. You need to confirm via `agent.confirm_memory_addition()`

In [17]:
# Use the trigger phrase to prompt memory storage
result = agent.run(
    "I just applied to a Senior Software Engineer position at Google. "
    "The role is focused on distributed systems and will be based in Mountain View. "
    "Save this to memory."
)
print(result.output)

Yes, please add it to memory.


In [18]:
# Check pending proposals
print("Pending proposals:", agent.pending_proposals.keys())

Pending proposals: dict_keys(['42ec8dea-4ed8-4247-a5f1-d1abfb936d83'])


In [19]:
# Confirm the first proposal (if any exist)
if agent.pending_proposals:
    proposal_id = list(agent.pending_proposals.keys())[0]
    result = agent.confirm_memory_addition(proposal_id, action="add_new")
    print(result)
else:
    print("No pending proposals")

Successfully added job to memory with ID: 9f57667f-21ae-43f6-baa6-bcb489602e3d


### Example 3: Searching Memory

In [20]:
# Test direct search with different thresholds
test_queries = [
    "software engineering positions",
    "Google job application",
    "Senior Software Engineer"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    
    # Try with lower threshold
    results = agent.store.search_similar_nodes(
        content=query,
        threshold=0.5,  # Lower threshold to see what we get
        limit=5,
        project_id="job_search_2026"
    )
    
    if results:
        for r in results:
            print(f"  Score: {r.similarity_score:.3f} | {r.content[:80]}...")
    else:
        print("  No results found")

# Now try with the agent
print("\n" + "="*50)
print("Agent search:")
result = agent.run("What software engineering positions have I applied to?")
print(result.output)


Query: 'software engineering positions'
--------------------------------------------------
  Score: 0.521 | Applied to a Senior Software Engineer position at Google focused on distributed ...

Query: 'Google job application'
--------------------------------------------------
  Score: 0.524 | Applied to a Senior Software Engineer position at Google focused on distributed ...

Query: 'Senior Software Engineer'
--------------------------------------------------
  Score: 0.578 | Applied to a Senior Software Engineer position at Google focused on distributed ...

Agent search:
You have applied to a Senior Software Engineer position at Google, which focuses on distributed systems and is based in Mountain View. Would you like to add more details about this or check for more applications?


### Example 4: Using Interval-Based Memory Trigger

In [21]:
# Change to interval mode - check every 5 messages
agent.set_memory_trigger(mode="interval", message_interval=5)
print("✓ Memory trigger set to: every 5 messages")

✓ Memory trigger set to: every 5 messages


In [22]:
# Send messages with meaningful content - on the 5th, agent should auto-check for memory items
messages = [
    "I had a phone screening with Meta today",
    "The recruiter mentioned the team works on AI infrastructure", 
    "They're looking for someone with distributed systems experience",
    "The position is remote-friendly with offices in Menlo Park",
    "I think this could be a good fit for my background"  # Message 5 - should trigger memory review
]

for i, msg in enumerate(messages, 1):
    result = agent.run(msg)
    print(f"Message {i}: {msg}")
    print(f"Response: {result.output}\n")

# Check if any proposals were created
if agent.pending_proposals:
    print(f"\n✓ Agent proposed {len(agent.pending_proposals)} memory addition(s)")
    print(f"Proposal IDs: {list(agent.pending_proposals.keys())}")
else:
    print("\n✗ No memory proposals generated")

Message 1: I had a phone screening with Meta today
Response: Should I add "Phone screening with Meta" to memory?

Message 2: The recruiter mentioned the team works on AI infrastructure
Response: I found similar content related to your job application at Google. Here are the proposals for your confirmation:

1. Should I add **"The recruiter mentioned the team works on AI infrastructure."** as an interaction to memory?
2. Should I add **"Google"** as a company to memory?

Please let me know how you'd like to proceed!

Message 3: They're looking for someone with distributed systems experience
Response: I found a related entry for a Senior Software Engineer position at Google that focuses on distributed systems. Would you like to update this existing memory or create a new entry?

Message 4: The position is remote-friendly with offices in Menlo Park
Response: I found a similar job application related to a Senior Software Engineer position at Google, which is based in Mountain View and like

### Example 5: Dynamic Configuration Updates

In [23]:
# Update memory configuration on the fly
agent.configure_memory(
    use_case_description="Track job search and networking contacts",
    memory_threshold_description="Store all interactions with recruiters and hiring managers",
    similarity_threshold=0.90,  # Stricter duplicate detection
)
print("✓ Memory configuration updated")

✓ Memory configuration updated


### Example 6: Viewing Audit Logs

In [24]:
# Get recent audit entries
audit_entries = agent.get_audit_history(limit=10)

print(f"Recent audit log entries ({len(audit_entries)}):")
for entry in audit_entries:
    print(f"\n[{entry.timestamp}] {entry.operation_type}")
    print(f"  Summary: {entry.summary}")
    print(f"  Entities: {entry.affected_entities}")

Recent audit log entries (5):

[2026-03-22 01:55:04.470571] add_node
  Summary: Added job: Applied to a Senior Software Engineer position at Google focused on distributed systems based in Mou
  Entities: ['4f0ad4fa-9606-44ee-b0be-7b84e85297c0']

[2026-03-22 01:56:42.333995] add_node
  Summary: Added job: Applied to Senior Software Engineer position at Google focused on distributed systems, based in Moun
  Entities: ['5a20cafc-5d1e-4e89-b7e6-5852f3d589ca']

[2026-03-22 02:01:31.746646] add_node
  Summary: Added job: Applied to a Senior Software Engineer position at Google focused on distributed systems, based in Mo
  Entities: ['caf12bb9-2d54-4825-b000-f057385f8d8a']

[2026-03-22 02:04:40.242716] add_node
  Summary: Added job: Applied to a Senior Software Engineer position at Google focused on distributed systems based in Mou
  Entities: ['a8b6948d-af92-4ef8-827a-4455076ad190']

[2026-03-22 02:06:59.376155] add_node
  Summary: Added job: Applied to a Senior Software Engineer position at

## Cleanup

In [25]:
# Close JanusGraph connection
janus.close()
print("✓ Database connections closed")

✓ Database connections closed
